# 4.2.1 Regularization for Deep Learning

L2 weight decay, Dropout, early stopping — training vs validation loss gap.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sigmoid=lambda x:1/(1+np.exp(-np.clip(x,-500,500)))
relu=lambda x:np.maximum(0,x); relu_d=lambda x:(x>0).astype(float)

def train(X_tr,y_tr,X_v,y_v,nh=64,lr=.05,ep=300,l2=0.,dr=0.,seed=42):
    rng=np.random.default_rng(seed)
    W1=rng.standard_normal((X_tr.shape[1],nh))*np.sqrt(2/X_tr.shape[1]); b1=np.zeros(nh)
    W2=rng.standard_normal((nh,1))*np.sqrt(2/nh); b2=np.zeros(1)
    tl,vl=[],[]
    for _ in range(ep):
        z1=X_tr@W1+b1; a1=relu(z1)
        mk=(rng.random(a1.shape)>dr)/(1-dr) if dr>0 else np.ones_like(a1); a1=a1*mk
        z2=a1@W2+b2; a2=sigmoid(z2); y2=y_tr.reshape(-1,1); n=len(y_tr)
        p=np.clip(a2,1e-7,1-1e-7); tl.append(-np.mean(y2*np.log(p)+(1-y2)*np.log(1-p)))
        dz2=(a2-y2)/n; dW2=a1.T@dz2+l2*W2/n; db2=dz2.sum(0)
        dz1=(dz2@W2.T)*relu_d(z1)*mk; dW1=X_tr.T@dz1+l2*W1/n; db1=dz1.sum(0)
        W2-=lr*dW2;b2-=lr*db2;W1-=lr*dW1;b1-=lr*db1
        z1v=X_v@W1+b1;a1v=relu(z1v);a2v=sigmoid(a1v@W2+b2);yv=y_v.reshape(-1,1)
        pv=np.clip(a2v,1e-7,1-1e-7); vl.append(-np.mean(yv*np.log(pv)+(1-yv)*np.log(1-pv)))
    return tl,vl

X,y=make_moons(800,noise=.3,random_state=42); Xs=StandardScaler().fit_transform(X)
Xt,Xv,yt,yv=train_test_split(Xs,y,test_size=.3,random_state=42)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(12,4))
for name,cfg in [('None',dict(l2=0,dr=0)),('L2=.01',dict(l2=.01,dr=0)),('Drop50',dict(l2=0,dr=.5))]:
    tl,vl=train(Xt,yt,Xv,yv,**cfg)
    ax[0].plot(tl,label=name); ax[1].plot(vl,label=name)
    print(f'{name}: val={vl[-1]:.4f}')
for a,t in zip(ax,['Train','Val']): a.legend(); a.set_title(t+' Loss'); a.set_ylim(0,1.5)
plt.tight_layout(); plt.show()